# 06 -- Expected Loss (EL = PD x LGD x EAD)

**What this notebook does (plain English):** Brings the three pieces together.
**Expected Loss** is the average loss a lender should budget for:

> **Expected Loss = chance of default (PD) x loss if it defaults (LGD) x amount
> owed (EAD)**

We score every loan, total it into a portfolio number, and sort loans into the
accounting **IFRS 9 / AASB 9 stages** (1 = healthy, 2 = deteriorating, 3 =
defaulted). We also walk through the full sum for one example loan.

**Headline result:** a single portfolio Expected Loss figure, dominated by the
Stage 3 (already-defaulted) loans and by the crisis vintages.

In [1]:
import sys, os
ROOT = os.getcwd()
if not os.path.isdir(os.path.join(ROOT, 'src')):
    ROOT = os.path.dirname(ROOT)
os.chdir(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
import warnings; warnings.filterwarnings('ignore')
print('project root:', ROOT)

project root: D:\Jane\Job Search\Github\bank\github project\freddie mac mortgage


In [2]:
# Load the base table and build the three components for EVERY loan.
import pandas as pd
import numpy as np
from src import models
from src.output import save_csv
base = pd.read_parquet('data/processed/analysis_base.parquet').copy()

In [3]:
# One-year PD for every loan (logistic model fit on the whole book). pd_hat is the
# raw, continuous 12-month model score (PD-1/PD-2), kept for ranking/comparison.
from src import definitions as d
pd_model, pd_cols = models.fit_pd(base)
base['pd_hat'] = d.apply_pd_floor(models.predict_pd(pd_model, pd_cols, base), floor=0.0005)
# PDR2-1: the PD that feeds EL must be the SAME PD as capital (EL framework Part 5.1) --
# the CALIBRATED grade PD (long-run + risk-sensitive MoC + ratchet + floor) from 03b.
grade_pd_map = pd.read_csv('output/03f_loan_grade_pd.csv')[
    ['loan_sequence_number', 'grade_pd_longrun', 'grade_pd_final']]
base['loan_sequence_number'] = base['loan_sequence_number'].astype(str)
grade_pd_map['loan_sequence_number'] = grade_pd_map['loan_sequence_number'].astype(str)
base = base.merge(grade_pd_map, on='loan_sequence_number', how='left')
base['pd_capital'] = base['grade_pd_final'].fillna(base['pd_hat'])
base['pd_longrun'] = base['grade_pd_longrun'].fillna(base['pd_hat'])

In [4]:
# LGD for every loan (two-stage model trained on disposed defaults).
disposed = base[base['disposed'] & base['lgd'].notna()]
lgd_model = models.TwoStageLGD().fit(disposed)
base['lgd_hat'] = lgd_model.predict(base)

In [5]:
# EAD for every loan: balance at default if it defaulted, else the original
# loan amount as the exposure proxy for a still-performing loan.
base['ead_loan'] = np.where(base['ever_default'], base['ead'], base['original_upb'])
# Expected loss per loan = PD x LGD x EAD, using the CALIBRATED capital PD (PDR2-1).
base['expected_loss'] = base['pd_capital'] * base['lgd_hat'] * base['ead_loan']

### The EL/capital PD reconcile (PDR2-1)

EL framework Part 5.1 requires Expected Loss to use the **same PD as capital/RWA**. Earlier
the dollar loss used only the raw model score with the 5 bps floor, so the long-run
calibration and the margin of conservatism never reached EL. Now `expected_loss` uses
**`pd_capital`** -- the calibrated grade PD (long-run average + risk-sensitive MoC + ratchet
+ floor) exported from notebook 03b -- so EL and the master-scale/capital PD are one and the
same number. The raw `pd_hat` is retained only for ranking and the comparison below.

In [6]:
# PDR2-1 impact: EL on three PD bases. The honest comparison is LIKE-FOR-LIKE at the
# pooled grade level (rows 2 vs 3): the MoC + ratchet flow straight through, so EL on the
# calibrated capital PD is the HIGHER (safer) one. Row 1 (the raw continuous score) is
# shown for context -- see the note on why pooling changes the level.
el_raw = float((base['pd_hat'] * base['lgd_hat'] * base['ead_loan']).sum())
el_lr = float((base['pd_longrun'] * base['lgd_hat'] * base['ead_loan']).sum())
el_cap = float(base['expected_loss'].sum())
pd_compare = pd.DataFrame([
    {'pd_basis': '1. raw continuous model PD (pre-calibration, floored)',
     'mean_pd': round(float(base['pd_hat'].mean()), 5), 'portfolio_EL': round(el_raw, 0)},
    {'pd_basis': '2. pooled grade PD, long-run only (no MoC)',
     'mean_pd': round(float(base['pd_longrun'].mean()), 5), 'portfolio_EL': round(el_lr, 0)},
    {'pd_basis': '3. calibrated capital PD (long-run + MoC + ratchet + floor)',
     'mean_pd': round(float(base['pd_capital'].mean()), 5), 'portfolio_EL': round(el_cap, 0)},
])
pd_compare['EL_vs_longrun_x'] = (pd_compare['portfolio_EL'] / el_lr).round(3)
save_csv(pd_compare, 'output/06_pd_basis_el_compare.csv')
print('MoC flows through (EL capital >= EL long-run, like-for-like):', el_cap >= el_lr)
pd_compare

MoC flows through (EL capital >= EL long-run, like-for-like): True


,pd_basis,mean_pd,portfolio_EL,EL_vs_longrun_x
0,"1. raw continuous model PD (pre-calibration, f...",0.00782,113340944.0,1.234
1,"2. pooled grade PD, long-run only (no MoC)",0.00727,91875710.0,1.000
2,3. calibrated capital PD (long-run + MoC + rat...,0.00832,105068726.0,1.144


**Reading the PD-basis comparison (PDR2-1).** On the **like-for-like pooled basis**
(rows 2 vs 3) the margin of conservatism and the grade-H ratchet flow straight through:
the calibrated **capital PD raises EL** above the bare long-run calibration -- the
conservatism now reaches the dollar loss, which was the whole gap this task closes.

Row 1 (the raw continuous score) actually sits a little **above** the calibrated capital
EL. That is **not** missing conservatism -- the mean calibrated PD is higher -- it is a
**pooling effect**: collapsing 150,000 continuous scores into 8 grade PDs removes the
within-grade correlation between the model score and exposure (the riskiest, largest loans
inside a grade no longer carry an individually higher PD). Capital frameworks pool exposures
into grades/pools by design (the grade PD *is* the regulatory PD), so the calibrated capital
EL is the correct figure to report, and it now reconciles exactly with the master scale.

In [7]:
# IFRS 9 / AASB 9 staging: 3 = defaulted (credit-impaired), 2 = significant
# increase in risk (ever 60+ days late but not defaulted), 1 = performing.
stage2 = (~base['ever_default']) & (base['max_delinq_status'].fillna(0) >= 2)
base['ifrs9_stage'] = np.where(base['ever_default'], 3, np.where(stage2, 2, 1))
# pd_capital is a genuine 12-MONTH PD, which is exactly the Stage 1 (12-month ECL)
# input -- so Stage 1 reported EL is the 12-month EL directly (no ad-hoc 0.25 factor
# any more). Stages 2 & 3 need LIFETIME ECL; with only a one-year PD modelled here we
# scale by a transparent multi-year horizon factor as a lifetime proxy (PDR2-7: a
# production model would estimate a lifetime PD term structure directly).
LIFETIME_HORIZON = 4
base['el_reported'] = np.where(base['ifrs9_stage'] == 1, base['expected_loss'],
                               base['expected_loss'] * LIFETIME_HORIZON)

In [8]:
# Portfolio Expected Loss summary by IFRS 9 stage (the saved result). With a
# one-year PD, `expected_loss_12m` is a 12-MONTH EL; `reported_expected_loss` applies
# the IFRS 9 staging (12-month for Stage 1, lifetime proxy for Stages 2 & 3).
el_summary = base.groupby('ifrs9_stage').agg(
    loans=('loan_sequence_number', 'size'),
    avg_pd=('pd_capital', 'mean'),
    avg_lgd=('lgd_hat', 'mean'),
    total_ead=('ead_loan', 'sum'),
    expected_loss_12m=('expected_loss', 'sum'),
    reported_expected_loss=('el_reported', 'sum'),
).reset_index().round(2)
save_csv(el_summary, 'output/06_expected_loss.csv')
el_summary

,ifrs9_stage,loans,avg_pd,avg_lgd,total_ead,expected_loss_12m,reported_expected_loss
0,1,132177,0.01,0.44,2.703702e+10,81600985.47,81600985.47
1,2,6067,0.01,0.50,1.140920e+09,6874900.24,27499600.97
2,3,11756,0.01,0.53,2.247190e+09,16592840.56,66371362.24


**Lifetime PD note (PDR2-7).** `expected_loss_12m` is a genuine **12-month** EL,
the correct IFRS 9 **Stage 1** input. The `reported_expected_loss` column then needs
**lifetime** ECL for Stages 2 and 3, and here we approximate it by scaling the 12-month
figure by a flat multi-year **horizon factor** (`LIFETIME_HORIZON = 4`). This is a
deliberate, named **proxy**: a production model would instead estimate a full **lifetime
PD term structure** (cumulative one-year PDs across the remaining life, conditioned on
age and macro path) rather than a single scalar. The proxy is kept here only to show the
staged-ECL shape end to end.

### Downturn-LGD variant of Expected Loss (P2-3)

Notebook 04 showed loss severity is strongly **cyclical** (~25% calm vs ~57% crisis).
APS 113 Att D LGD paras 4-5 say that where severity is cyclical, the LGD *estimate*
must reflect **downturn** conditions, not the through-the-cycle average. So alongside
the baseline EL we compute a **downturn-LGD variant**, lifting every loan's LGD to at
least the crisis-regime realised severity. This is the conservative figure the
framework expects a capital/EL report to show.

In [9]:
# P2-3: downturn-LGD variant of EL. Lift each loan's LGD to >= the crisis-regime
# realised severity (the observed downturn LGD), then recompute Expected Loss.
downturn_lgd = float(base.loc[base['disposed'] & base['vintage_year'].isin([2007, 2008]), 'lgd'].mean())
base['lgd_downturn'] = np.maximum(base['lgd_hat'], downturn_lgd)
base['expected_loss_downturn'] = base['pd_capital'] * base['lgd_downturn'] * base['ead_loan']
el_variant = pd.DataFrame([
    {'view': 'through-the-cycle (baseline)', 'lgd_basis': 'modelled lgd_hat',
     'total_expected_loss': round(float(base['expected_loss'].sum()), 0)},
    {'view': 'downturn LGD (APS 113 Att D LGD 4-5)', 'lgd_basis': f'max(lgd_hat, {downturn_lgd:.3f})',
     'total_expected_loss': round(float(base['expected_loss_downturn'].sum()), 0)},
])
el_variant['uplift_x'] = (el_variant['total_expected_loss'] /
                          el_variant['total_expected_loss'].iloc[0]).round(2)
save_csv(el_variant, 'output/06_el_downturn_variant.csv')
el_variant

,view,lgd_basis,total_expected_loss,uplift_x
0,through-the-cycle (baseline),modelled lgd_hat,105068726.0,1.00
1,downturn LGD (APS 113 Att D LGD 4-5),"max(lgd_hat, 0.567)",143361483.0,1.36


### Best estimate of EL for already-defaulted (Stage 3) loans (P2-4)

APS 113 Att D para 11 / Part 4.3: for loans **already in default** (Stage 3), you must
form a **best estimate of expected loss for that loan given current conditions** --
mechanically applying the model's average LGD is "not acceptable". We replace the
mechanical PD x LGD with: the loan's **realised** LGD where its workout is materially
complete (disposed), otherwise the **segment downturn LGD**; since the loan is already
in default its PD is 1, so EL = best-estimate LGD x EAD.

In [10]:
# P2-4: best-estimate EL for Stage 3 (already-defaulted) loans.
stage3 = base['ifrs9_stage'] == 3
best_lgd = np.where(base['disposed'] & base['lgd'].notna(), base['lgd'], downturn_lgd)
base['el_stage3_bestestimate'] = np.where(stage3, best_lgd * base['ead_loan'], np.nan)
s3 = pd.DataFrame([{
    'stage3_loans': int(stage3.sum()),
    'el_mechanical_pd_x_lgd': round(float(base.loc[stage3, 'expected_loss'].sum()), 0),
    'el_best_estimate': round(float(np.nansum(base['el_stage3_bestestimate'])), 0),
}])
s3['ratio_best_vs_mechanical'] = round(s3['el_best_estimate'] / s3['el_mechanical_pd_x_lgd'], 2)
save_csv(s3, 'output/06_stage3_best_estimate.csv')
s3

,stage3_loans,el_mechanical_pd_x_lgd,el_best_estimate,ratio_best_vs_mechanical
0,11756,16592841.0,1.223179e+09,73.72


**Reading the Stage 3 table (P2-4).** The mechanical column applies the model
PD x LGD even to loans that have *already* defaulted (so its PD < 1 understates the
loss); the best-estimate column uses each defaulted loan's realised loss where the
workout is complete and the downturn LGD otherwise, with PD = 1. The best estimate is
materially larger -- which is the point: a defaulted loan's expected loss should be
built from its own resolution, not a portfolio-average model output.

In [11]:
# Worked example: show PD x LGD x EAD = EL for a single representative loan,
# using the calibrated capital PD (the same PD the portfolio EL is built on).
ex = base.sort_values('expected_loss', ascending=False).iloc[100]
print('Worked example loan:', ex['loan_sequence_number'])
print(f"  PD  (calibrated 1yr)    = {ex['pd_capital']:.3f}")
print(f"  LGD (loss if default)   = {ex['lgd_hat']:.3f}")
print(f"  EAD (amount owed)       = ${ex['ead_loan']:,.0f}")
print(f"  Expected Loss = {ex['pd_capital']:.3f} x {ex['lgd_hat']:.3f} x ${ex['ead_loan']:,.0f} = ${ex['expected_loss']:,.0f}")

Worked example loan: F07Q10374715
  PD  (calibrated 1yr)    = 0.027
  LGD (loss if default)   = 0.463
  EAD (amount owed)       = $411,000
  Expected Loss = 0.027 x 0.463 x $411,000 = $5,082


**Reading the table:** Stage 3 holds the already-defaulted loans and
carries most of the loss; Stage 1 is the large healthy book on a 12-month view.
The worked example shows the headline equation end-to-end for one loan.